# 🎤 InterviewCoach — ASR Model Comparison & Fine-Tuning

**Objective**: Compare 4 offline ASR models on English and Indonesian interview speech, fine-tune the best one, and export a `model_config.json` for the Streamlit app.

| # | Model | Library | Notes |
|---|-------|---------|-------|
| 1 | **Vosk** | `vosk` | Kaldi-based, real-time capable |
| 2 | **Whisper** | `openai-whisper` | OpenAI multilingual, offline |
| 3 | **Wav2Vec2** | `transformers` | Facebook CTC, HuggingFace |
| 4 | **SpeechBrain** | `speechbrain` | End-to-end neural ASR |

**Metrics**: WER, CER, RTF (Real-Time Factor), Accuracy  
**Datasets**: 10 EN (HuggingFace multispeak) + 10 ID (Kaggle)

---
## 📦 1. Setup & Installation

In [ ]:
import subprocess, sys

packages = [
    "vosk",
    "openai-whisper",
    "transformers>=4.30",
    "datasets",
    "torch",
    "torchaudio",
    "soundfile",
    "librosa",
    "jiwer",
    "speechbrain",
    "huggingface_hub",
    "matplotlib",
    "seaborn",
    "pandas",
    "numpy",
    "scipy",
    "tqdm",
    "kaggle",
]

for pkg in packages:
    print(f"Installing {pkg}...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("\n✅ All packages installed!")

---
## 📥 2. Imports & Configuration

In [ ]:
import os
import io
import json
import time
import wave
import zipfile
import logging
import tempfile
import urllib.request
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import librosa
import soundfile as sf

logging.basicConfig(level=logging.WARNING)

# ── Paths ──────────────────────────────────────────────────
MODELS_DIR  = Path("models")
DATA_DIR    = Path("data")
DATA_EN_DIR = DATA_DIR / "english"
DATA_ID_DIR = DATA_DIR / "indonesian"
OUTPUT_DIR  = Path("outputs")

for d in [MODELS_DIR, DATA_EN_DIR, DATA_ID_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SAMPLE_RATE = 16000   # all models expect 16 kHz mono

print("Directories ready.")
print(f"  models/ : {MODELS_DIR.resolve()}")
print(f"  data/   : {DATA_DIR.resolve()}")
print(f"  outputs/: {OUTPUT_DIR.resolve()}")

---
## 🇺🇸 3. English Dataset — HuggingFace `multispeak/hiring-accent-speech-human-voices`

In [ ]:
from datasets import load_dataset

print("Loading English dataset from HuggingFace...")
ds_en = load_dataset(
    "multispeak/hiring-accent-speech-human-voices",
    split="train",
    streaming=False,
)

print(f"Total samples available: {len(ds_en)}")
print(f"Columns: {ds_en.column_names}")

# Take first 10 samples
EN_SAMPLES = []
for i, sample in enumerate(ds_en):
    if i >= 10:
        break
    EN_SAMPLES.append(sample)

print(f"\n✅ Loaded {len(EN_SAMPLES)} English samples.")
print("\nSample fields:", list(EN_SAMPLES[0].keys()) if EN_SAMPLES else "(none)")

In [ ]:
EN_AUDIO_PATHS   = []
EN_TRANSCRIPTS   = []

for i, sample in enumerate(EN_SAMPLES):
    # HuggingFace Audio datasets typically have 'audio' (dict with 'array', 'sampling_rate')
    # and a text/transcript column (commonly 'text' or 'transcript' or 'sentence')
    audio_info = sample.get("audio", {})
    audio_arr  = np.array(audio_info.get("array", []), dtype=np.float32)
    sr         = audio_info.get("sampling_rate", SAMPLE_RATE)

    # Resample to 16 kHz if needed
    if sr != SAMPLE_RATE and len(audio_arr) > 0:
        audio_arr = librosa.resample(audio_arr, orig_sr=sr, target_sr=SAMPLE_RATE)

    # Get transcript — try common column names
    transcript = (
        sample.get("text") or
        sample.get("transcript") or
        sample.get("sentence") or
        sample.get("transcription") or
        "[no transcript]"
    )
    if isinstance(transcript, str):
        transcript = transcript.strip()

    # Save WAV
    wav_path = DATA_EN_DIR / f"en_sample_{i:02d}.wav"
    sf.write(str(wav_path), audio_arr, SAMPLE_RATE)

    EN_AUDIO_PATHS.append(str(wav_path))
    EN_TRANSCRIPTS.append(transcript)
    print(f"  [{i+1}] saved {wav_path.name}  |  transcript: {transcript[:60]}...")

print(f"\n✅ {len(EN_AUDIO_PATHS)} English audio files saved.")

---
## 🇮🇩 4. Indonesian Dataset — Kaggle `kambingbersayaphitam/speech-dataset-of-human-and-ai-generated-voices`

In [ ]:
# ── Option A: download via Kaggle API (requires ~/.kaggle/kaggle.json) ──
# If you have credentials:
#   kaggle datasets download -d kambingbersayaphitam/speech-dataset-of-human-and-ai-generated-voices
#   unzip to data/indonesian/

# ── Option B: HuggingFace mirror search ──
# Try loading an Indonesian ASR dataset from HuggingFace as fallback

ID_AUDIO_PATHS  = []
ID_TRANSCRIPTS  = []

# Check if already downloaded manually
existing_wavs = sorted(DATA_ID_DIR.glob("*.wav"))[:10]
if existing_wavs:
    print(f"Found {len(existing_wavs)} existing Indonesian WAV files.")
    for p in existing_wavs:
        ID_AUDIO_PATHS.append(str(p))
        # Try to load transcript from matching .txt file
        txt_p = p.with_suffix(".txt")
        transcript = txt_p.read_text(encoding="utf-8").strip() if txt_p.exists() else "[no transcript]"
        ID_TRANSCRIPTS.append(transcript)
else:
    print("No local Indonesian data found. Using HuggingFace fallback...")
    try:
        # Fallback: indonesian common voice
        ds_id = load_dataset("mozilla-foundation/common_voice_11_0",
                             "id", split="test", streaming=False, trust_remote_code=True)
        for i, sample in enumerate(ds_id):
            if i >= 10:
                break
            audio_info = sample.get("audio", {})
            audio_arr  = np.array(audio_info.get("array", []), dtype=np.float32)
            sr         = audio_info.get("sampling_rate", SAMPLE_RATE)
            if sr != SAMPLE_RATE and len(audio_arr) > 0:
                audio_arr = librosa.resample(audio_arr, orig_sr=sr, target_sr=SAMPLE_RATE)
            transcript = sample.get("sentence", "[no transcript]").strip()
            wav_path   = DATA_ID_DIR / f"id_sample_{i:02d}.wav"
            sf.write(str(wav_path), audio_arr, SAMPLE_RATE)
            ID_AUDIO_PATHS.append(str(wav_path))
            ID_TRANSCRIPTS.append(transcript)
            print(f"  [{i+1}] {wav_path.name}  |  {transcript[:60]}...")
        print(f"\n✅ {len(ID_AUDIO_PATHS)} Indonesian samples loaded (Common Voice fallback).")
    except Exception as e:
        print(f"Fallback also failed: {e}")
        print("Generating synthetic Indonesian placeholders...")
        # Generate silent placeholder WAVs with dummy text for testing
        ID_PLACEHOLDER_TEXTS = [
            "Saya memiliki pengalaman bekerja selama lima tahun di bidang teknologi.",
            "Kelebihan terbesar saya adalah kemampuan komunikasi dan kerja tim.",
            "Saya melihat diri saya sebagai pemimpin tim dalam lima tahun ke depan.",
            "Tantangan terbesar saya adalah mengelola proyek dengan tenggat waktu ketat.",
            "Saya termotivasi oleh pencapaian target dan pertumbuhan karir.",
            "Gaya kepemimpinan saya bersifat kolaboratif dan berorientasi hasil.",
            "Saya selalu memastikan kualitas pekerjaan melalui proses review yang ketat.",
            "Pengalaman terbaik saya adalah memimpin tim lintas departemen.",
            "Saya beradaptasi dengan perubahan melalui komunikasi yang terbuka.",
            "Rekan kerja saya menggambarkan saya sebagai pribadi yang dapat diandalkan.",
        ]
        for i, text in enumerate(ID_PLACEHOLDER_TEXTS):
            # 3-second silent audio
            silence = np.zeros(SAMPLE_RATE * 3, dtype=np.float32)
            wav_path = DATA_ID_DIR / f"id_placeholder_{i:02d}.wav"
            sf.write(str(wav_path), silence, SAMPLE_RATE)
            ID_AUDIO_PATHS.append(str(wav_path))
            ID_TRANSCRIPTS.append(text)
        print(f"Created {len(ID_AUDIO_PATHS)} placeholder files. Replace with real recordings!")

print(f"\nTotal Indonesian samples: {len(ID_AUDIO_PATHS)}")

---
## 🔊 5. Audio Preprocessing & Visualization

In [ ]:
def load_and_normalize(wav_path: str, target_sr: int = 16000) -> Tuple[np.ndarray, int]:
    """Load WAV, resample, normalize to [-1, 1]."""
    audio, sr = librosa.load(wav_path, sr=target_sr, mono=True)
    # Peak normalize
    peak = np.max(np.abs(audio))
    if peak > 0:
        audio = audio / peak * 0.95
    return audio.astype(np.float32), target_sr


def plot_waveform_spectrogram(wav_path: str, title: str = ""):
    """Plot waveform + mel spectrogram side by side."""
    audio, sr = load_and_normalize(wav_path)
    fig, axes = plt.subplots(1, 2, figsize=(14, 3))

    # Waveform
    t = np.linspace(0, len(audio) / sr, len(audio))
    axes[0].plot(t, audio, color="#6366f1", linewidth=0.6)
    axes[0].set_title(f"Waveform — {title}", fontsize=11)
    axes[0].set_xlabel("Time (s)"); axes[0].set_ylabel("Amplitude")
    axes[0].grid(alpha=0.3)

    # Mel spectrogram
    mel = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=80)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    img = librosa.display.specshow(mel_db, sr=sr, x_axis="time",
                                    y_axis="mel", ax=axes[1], cmap="magma")
    fig.colorbar(img, ax=axes[1], format="%+2.0f dB")
    axes[1].set_title(f"Mel Spectrogram — {title}", fontsize=11)

    plt.tight_layout()
    plt.savefig(str(OUTPUT_DIR / f"viz_{title.replace(' ', '_')}.png"), dpi=100)
    plt.show()


# Visualize one English and one Indonesian sample
if EN_AUDIO_PATHS:
    plot_waveform_spectrogram(EN_AUDIO_PATHS[0], "EN Sample 0")
if ID_AUDIO_PATHS:
    plot_waveform_spectrogram(ID_AUDIO_PATHS[0], "ID Sample 0")

# Duration stats
all_paths = EN_AUDIO_PATHS + ID_AUDIO_PATHS
durations = [librosa.get_duration(filename=p) for p in all_paths if Path(p).exists()]
print(f"\nAudio duration stats (all {len(durations)} samples):")
print(f"  Min: {min(durations):.1f}s")
print(f"  Max: {max(durations):.1f}s")
print(f"  Avg: {np.mean(durations):.1f}s")
print(f"  Total: {sum(durations):.1f}s ({sum(durations)/60:.1f} min)")

---
## 📐 Metric Helpers

In [ ]:
from jiwer import wer as compute_wer, cer as compute_cer


def evaluate(ref_texts: List[str], hyp_texts: List[str]) -> Dict:
    """Compute WER, CER, Accuracy over a list of utterances."""
    # Filter empty refs
    pairs = [(r.lower().strip(), h.lower().strip())
             for r, h in zip(ref_texts, hyp_texts)
             if r and r != "[no transcript]"]
    if not pairs:
        return {"wer": None, "cer": None, "accuracy": None, "samples": 0}

    refs, hyps = zip(*pairs)
    wer_val = compute_wer(list(refs), list(hyps))
    cer_val = compute_cer(list(refs), list(hyps))
    # Word accuracy = 1 - WER  (clamped to [0,1])
    acc = max(0.0, 1.0 - wer_val)

    return {
        "wer":      round(wer_val * 100, 2),   # %
        "cer":      round(cer_val * 100, 2),   # %
        "accuracy": round(acc * 100, 2),       # %
        "samples":  len(pairs),
    }


def measure_rtf(start: float, end: float, audio_duration: float) -> float:
    """Real-Time Factor = processing_time / audio_duration."""
    proc = end - start
    if audio_duration <= 0:
        return float("inf")
    return round(proc / audio_duration, 3)


print("Metric helpers ready.")

---
## 🔵 Model 1: Vosk

In [ ]:
import vosk
vosk.SetLogLevel(-1)

VOSK_MODELS = {
    "en": {"dir": "models/vosk-model-en-us-0.22",
           "url": "https://alphacephei.com/vosk/models/vosk-model-en-us-0.22.zip"},
    "id": {"dir": "models/vosk-model-id-0.21",
           "url": "https://alphacephei.com/vosk/models/vosk-model-id-0.21.zip"},
}


def download_vosk_model(lang: str):
    info = VOSK_MODELS[lang]
    model_dir = Path(info["dir"])
    if model_dir.exists():
        print(f"Vosk {lang} model already present at {model_dir}")
        return
    model_dir.parent.mkdir(parents=True, exist_ok=True)
    zip_path = str(model_dir) + ".zip"
    print(f"Downloading Vosk {lang} model from {info['url']} ...")
    urllib.request.urlretrieve(info["url"], zip_path)
    print("Extracting...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(str(model_dir.parent))
    os.remove(zip_path)
    print(f"✅ Vosk {lang} model ready.")


def transcribe_vosk(wav_path: str, model: vosk.Model) -> str:
    rec = vosk.KaldiRecognizer(model, SAMPLE_RATE)
    rec.SetWords(True)
    with wave.open(wav_path, "rb") as wf:
        while True:
            data = wf.readframes(4000)
            if not data:
                break
            rec.AcceptWaveform(data)
    result = json.loads(rec.FinalResult())
    return result.get("text", "")


# Download models
download_vosk_model("en")
download_vosk_model("id")

# Load
print("Loading Vosk EN model...")
vosk_model_en = vosk.Model(VOSK_MODELS["en"]["dir"])
print("Loading Vosk ID model...")
vosk_model_id = vosk.Model(VOSK_MODELS["id"]["dir"])
print("✅ Both Vosk models loaded.")

In [ ]:
print("=== Vosk Inference ===")
vosk_hyps_en = []
vosk_rtfs_en = []

for path, ref in zip(EN_AUDIO_PATHS, EN_TRANSCRIPTS):
    dur   = librosa.get_duration(filename=path)
    t0    = time.perf_counter()
    hyp   = transcribe_vosk(path, vosk_model_en)
    rtf   = measure_rtf(t0, time.perf_counter(), dur)
    vosk_hyps_en.append(hyp)
    vosk_rtfs_en.append(rtf)
    print(f"  EN ref: {ref[:50]}")
    print(f"     hyp: {hyp[:50]}  | RTF={rtf}")

vosk_hyps_id = []
vosk_rtfs_id = []

for path, ref in zip(ID_AUDIO_PATHS, ID_TRANSCRIPTS):
    dur = librosa.get_duration(filename=path)
    t0  = time.perf_counter()
    hyp = transcribe_vosk(path, vosk_model_id)
    rtf = measure_rtf(t0, time.perf_counter(), dur)
    vosk_hyps_id.append(hyp)
    vosk_rtfs_id.append(rtf)
    print(f"  ID ref: {ref[:50]}")
    print(f"     hyp: {hyp[:50]}  | RTF={rtf}")

vosk_metrics_en = evaluate(EN_TRANSCRIPTS, vosk_hyps_en)
vosk_metrics_id = evaluate(ID_TRANSCRIPTS, vosk_hyps_id)
vosk_metrics_en["avg_rtf"] = round(np.mean(vosk_rtfs_en), 3)
vosk_metrics_id["avg_rtf"] = round(np.mean(vosk_rtfs_id), 3)

print("\n📊 Vosk EN:", vosk_metrics_en)
print("📊 Vosk ID:", vosk_metrics_id)

---
## 🟠 Model 2: Whisper

In [ ]:
import whisper

print("Loading Whisper 'base' model...")
whisper_model = whisper.load_model("base")
print("✅ Whisper loaded.")


def transcribe_whisper(wav_path: str, language: Optional[str] = None) -> str:
    opts = {}
    if language:
        opts["language"] = language
    result = whisper_model.transcribe(wav_path, **opts)
    return result.get("text", "").strip()

In [ ]:
print("=== Whisper Inference ===")
whisper_hyps_en, whisper_rtfs_en = [], []
for path, ref in zip(EN_AUDIO_PATHS, EN_TRANSCRIPTS):
    dur = librosa.get_duration(filename=path)
    t0  = time.perf_counter()
    hyp = transcribe_whisper(path, language="en")
    rtf = measure_rtf(t0, time.perf_counter(), dur)
    whisper_hyps_en.append(hyp)
    whisper_rtfs_en.append(rtf)
    print(f"  EN ref: {ref[:50]}")
    print(f"     hyp: {hyp[:50]}  | RTF={rtf}")

whisper_hyps_id, whisper_rtfs_id = [], []
for path, ref in zip(ID_AUDIO_PATHS, ID_TRANSCRIPTS):
    dur = librosa.get_duration(filename=path)
    t0  = time.perf_counter()
    hyp = transcribe_whisper(path, language="id")
    rtf = measure_rtf(t0, time.perf_counter(), dur)
    whisper_hyps_id.append(hyp)
    whisper_rtfs_id.append(rtf)
    print(f"  ID ref: {ref[:50]}")
    print(f"     hyp: {hyp[:50]}  | RTF={rtf}")

whisper_metrics_en = evaluate(EN_TRANSCRIPTS, whisper_hyps_en)
whisper_metrics_id = evaluate(ID_TRANSCRIPTS, whisper_hyps_id)
whisper_metrics_en["avg_rtf"] = round(np.mean(whisper_rtfs_en), 3)
whisper_metrics_id["avg_rtf"] = round(np.mean(whisper_rtfs_id), 3)

print("\n📊 Whisper EN:", whisper_metrics_en)
print("📊 Whisper ID:", whisper_metrics_id)

---
## 🟢 Model 3: Wav2Vec2 (HuggingFace Transformers)

In [ ]:
from transformers import pipeline as hf_pipeline
import torch

device = 0 if torch.cuda.is_available() else -1
print(f"Device: {'GPU' if device == 0 else 'CPU'}")

print("Loading Wav2Vec2 EN (facebook/wav2vec2-base-960h)...")
try:
    wav2vec2_en_pipe = hf_pipeline(
        "automatic-speech-recognition",
        model="facebook/wav2vec2-base-960h",
        device=device,
    )
    print("✅ Wav2Vec2 EN loaded.")
except Exception as e:
    print(f"❌ Wav2Vec2 EN load error: {e}")
    wav2vec2_en_pipe = None

print("Loading Wav2Vec2 ID (indonesian-nlp/wav2vec2-indonesian-javanese-sundanese)...")
try:
    wav2vec2_id_pipe = hf_pipeline(
        "automatic-speech-recognition",
        model="indonesian-nlp/wav2vec2-indonesian-javanese-sundanese",
        device=device,
    )
    print("✅ Wav2Vec2 ID loaded.")
except Exception as e:
    print(f"⚠️ Wav2Vec2 ID load error: {e}. Using EN model as fallback.")
    wav2vec2_id_pipe = wav2vec2_en_pipe


def transcribe_wav2vec2(wav_path: str, pipe) -> str:
    if pipe is None:
        return ""
    result = pipe(wav_path)
    return result.get("text", "").strip() if isinstance(result, dict) else ""

In [ ]:
print("=== Wav2Vec2 Inference ===")
w2v_hyps_en, w2v_rtfs_en = [], []
for path, ref in zip(EN_AUDIO_PATHS, EN_TRANSCRIPTS):
    dur = librosa.get_duration(filename=path)
    t0  = time.perf_counter()
    hyp = transcribe_wav2vec2(path, wav2vec2_en_pipe)
    rtf = measure_rtf(t0, time.perf_counter(), dur)
    w2v_hyps_en.append(hyp)
    w2v_rtfs_en.append(rtf)
    print(f"  EN ref: {ref[:50]}")
    print(f"     hyp: {hyp[:50]}  | RTF={rtf}")

w2v_hyps_id, w2v_rtfs_id = [], []
for path, ref in zip(ID_AUDIO_PATHS, ID_TRANSCRIPTS):
    dur = librosa.get_duration(filename=path)
    t0  = time.perf_counter()
    hyp = transcribe_wav2vec2(path, wav2vec2_id_pipe)
    rtf = measure_rtf(t0, time.perf_counter(), dur)
    w2v_hyps_id.append(hyp)
    w2v_rtfs_id.append(rtf)

w2v_metrics_en = evaluate(EN_TRANSCRIPTS, w2v_hyps_en)
w2v_metrics_id = evaluate(ID_TRANSCRIPTS, w2v_hyps_id)
w2v_metrics_en["avg_rtf"] = round(np.mean(w2v_rtfs_en), 3)
w2v_metrics_id["avg_rtf"] = round(np.mean(w2v_rtfs_id), 3)

print("\n📊 Wav2Vec2 EN:", w2v_metrics_en)
print("📊 Wav2Vec2 ID:", w2v_metrics_id)

---
## 🔴 Model 4: SpeechBrain

In [ ]:
try:
    from speechbrain.inference.ASR import EncoderDecoderASR
    SB_DIR = str(MODELS_DIR / "speechbrain")
    Path(SB_DIR).mkdir(parents=True, exist_ok=True)

    print("Loading SpeechBrain (speechbrain/asr-crdnn-rnnlm-librispeech)...")
    sb_model = EncoderDecoderASR.from_hparams(
        source="speechbrain/asr-crdnn-rnnlm-librispeech",
        savedir=SB_DIR,
        run_opts={"device": "cpu"},
    )
    print("✅ SpeechBrain loaded.")

    def transcribe_speechbrain(wav_path: str) -> str:
        try:
            return sb_model.transcribe_file(wav_path).strip()
        except Exception as e:
            print(f"  SpeechBrain error on {wav_path}: {e}")
            return ""

    SB_AVAILABLE = True
except Exception as e:
    print(f"❌ SpeechBrain load error: {e}")
    SB_AVAILABLE = False
    sb_model = None

    def transcribe_speechbrain(wav_path: str) -> str:
        return ""

In [ ]:
print("=== SpeechBrain Inference ===")
sb_hyps_en, sb_rtfs_en = [], []
for path, ref in zip(EN_AUDIO_PATHS, EN_TRANSCRIPTS):
    dur = librosa.get_duration(filename=path)
    t0  = time.perf_counter()
    hyp = transcribe_speechbrain(path)
    rtf = measure_rtf(t0, time.perf_counter(), dur)
    sb_hyps_en.append(hyp)
    sb_rtfs_en.append(rtf)
    print(f"  EN ref: {ref[:50]}")
    print(f"     hyp: {hyp[:50]}  | RTF={rtf}")

# SpeechBrain is English-only by default; reuse EN for ID test
sb_hyps_id, sb_rtfs_id = [], []
for path, ref in zip(ID_AUDIO_PATHS, ID_TRANSCRIPTS):
    dur = librosa.get_duration(filename=path)
    t0  = time.perf_counter()
    hyp = transcribe_speechbrain(path)  # EN model on ID audio — will score poorly
    rtf = measure_rtf(t0, time.perf_counter(), dur)
    sb_hyps_id.append(hyp)
    sb_rtfs_id.append(rtf)

sb_metrics_en = evaluate(EN_TRANSCRIPTS, sb_hyps_en)
sb_metrics_id = evaluate(ID_TRANSCRIPTS, sb_hyps_id)
sb_metrics_en["avg_rtf"] = round(np.mean(sb_rtfs_en), 3)
sb_metrics_id["avg_rtf"] = round(np.mean(sb_rtfs_id), 3)

print("\n📊 SpeechBrain EN:", sb_metrics_en)
print("📊 SpeechBrain ID:", sb_metrics_id)

---
## 📊 6. Model Comparison & Visualization

In [ ]:
# Build comparison dataframe
models = ["Vosk", "Whisper", "Wav2Vec2", "SpeechBrain"]

en_metrics_list = [vosk_metrics_en, whisper_metrics_en, w2v_metrics_en, sb_metrics_en]
id_metrics_list = [vosk_metrics_id, whisper_metrics_id, w2v_metrics_id, sb_metrics_id]

rows = []
for model, en_m, id_m in zip(models, en_metrics_list, id_metrics_list):
    rows.append({
        "Model": model,
        "Lang": "EN",
        "WER (%)": en_m.get("wer"),
        "CER (%)": en_m.get("cer"),
        "Accuracy (%)": en_m.get("accuracy"),
        "Avg RTF": en_m.get("avg_rtf"),
        "Samples": en_m.get("samples"),
    })
    rows.append({
        "Model": model,
        "Lang": "ID",
        "WER (%)": id_m.get("wer"),
        "CER (%)": id_m.get("cer"),
        "Accuracy (%)": id_m.get("accuracy"),
        "Avg RTF": id_m.get("avg_rtf"),
        "Samples": id_m.get("samples"),
    })

df = pd.DataFrame(rows)
print("\n📊 Full Comparison Table:")
print(df.to_string(index=False))

In [ ]:
# ── Bar charts ──────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
metrics_to_plot = ["WER (%)", "CER (%)", "Accuracy (%)", "Avg RTF"]
colors_en = "#6366f1"
colors_id = "#f59e0b"
bar_width  = 0.35
x = np.arange(len(models))

for ax, metric in zip(axes.flat, metrics_to_plot):
    en_vals = [df[(df.Model == m) & (df.Lang == "EN")][metric].values[0] for m in models]
    id_vals = [df[(df.Model == m) & (df.Lang == "ID")][metric].values[0] for m in models]

    # Replace None with 0
    en_vals = [v if v is not None else 0 for v in en_vals]
    id_vals = [v if v is not None else 0 for v in id_vals]

    bars1 = ax.bar(x - bar_width/2, en_vals, bar_width, label="EN", color=colors_en, alpha=0.85)
    bars2 = ax.bar(x + bar_width/2, id_vals, bar_width, label="ID", color=colors_id, alpha=0.85)
    ax.set_title(metric, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels(models)
    ax.legend()
    ax.grid(axis="y", alpha=0.3)

    # Annotate
    for bar in bars1:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.3, f"{h:.1f}",
                ha="center", fontsize=8)
    for bar in bars2:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.3, f"{h:.1f}",
                ha="center", fontsize=8)

plt.suptitle("InterviewCoach — ASR Model Comparison (EN vs ID)",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "model_comparison.png"), dpi=120, bbox_inches="tight")
plt.show()
print("Saved: outputs/model_comparison.png")

In [ ]:
# ── Select best model (lowest average WER across EN + ID) ──
avg_wers = {}
for model, en_m, id_m in zip(models, en_metrics_list, id_metrics_list):
    wer_en = en_m.get("wer") or 100.0
    wer_id = id_m.get("wer") or 100.0
    avg_wers[model] = (wer_en + wer_id) / 2

print("\n📉 Average WER (EN + ID combined):")
for m, v in sorted(avg_wers.items(), key=lambda x: x[1]):
    print(f"  {m:12s}: {v:.2f}%")

best_model_name = min(avg_wers, key=avg_wers.get)
best_model_key  = best_model_name.lower().replace("speech", "speech").replace(
    "speechbrain", "speechbrain"
)  # match keys in model_manager.py
best_model_key  = best_model_name.lower()

print(f"\n🏆 Best model: {best_model_name} (avg WER: {avg_wers[best_model_name]:.2f}%)")

---
## 🔧 7. Fine-Tuning the Best Model (Whisper)

> **Strategy**: We use Whisper's built-in multilingual capability and adapt it with our small dataset using gradient-based fine-tuning via the HuggingFace `Trainer`.  
> For Vosk, we would need re-training Kaldi — impractical on small data.  
> For Wav2Vec2, CTC fine-tuning is shown as an alternative below.

In [ ]:
import torch
from transformers import (
    WhisperForConditionalGeneration,
    WhisperProcessor,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
from datasets import Dataset, DatasetDict
import evaluate as hf_evaluate

FINETUNE_MODEL_DIR = str(MODELS_DIR / "whisper-finetuned")
Path(FINETUNE_MODEL_DIR).mkdir(parents=True, exist_ok=True)

# ── Load processor & model ──
print("Loading Whisper processor and model for fine-tuning...")
processor  = WhisperProcessor.from_pretrained("openai/whisper-base")
ft_model   = WhisperForConditionalGeneration.from_pretrained("openai/whisper-base")
ft_model.config.forced_decoder_ids = None
print("✅ Whisper base loaded for fine-tuning.")

# ── Build combined dataset ──
all_paths = EN_AUDIO_PATHS + ID_AUDIO_PATHS
all_refs  = EN_TRANSCRIPTS + ID_TRANSCRIPTS

def build_hf_dataset(paths, refs):
    """Build HuggingFace Dataset from local WAV files + transcripts."""
    records = []
    for path, ref in zip(paths, refs):
        if not Path(path).exists():
            continue
        audio, sr = librosa.load(path, sr=SAMPLE_RATE, mono=True)
        records.append({"audio": audio, "sampling_rate": sr, "text": ref})
    return Dataset.from_list(records)


def prepare_dataset(batch):
    """Feature-extract audio + tokenize labels."""
    audio_arr = batch["audio"]
    inputs = processor.feature_extractor(
        raw_speech=audio_arr,
        sampling_rate=SAMPLE_RATE,
        return_tensors="pt",
    )
    batch["input_features"] = inputs.input_features[0]

    with processor.as_target_processor():
        labels = processor.tokenizer(
            batch["text"], return_tensors="pt"
        )
    # Replace padding token id's of the labels by -100
    label_ids = labels.input_ids[0]
    label_ids[label_ids == processor.tokenizer.pad_token_id] = -100
    batch["labels"] = label_ids
    return batch


print("Building dataset...")
full_ds = build_hf_dataset(all_paths, all_refs)

# 80/20 train/eval split
split   = full_ds.train_test_split(test_size=0.2, seed=42)
train_ds = split["train"].map(prepare_dataset, remove_columns=["audio", "sampling_rate", "text"])
eval_ds  = split["test"].map(prepare_dataset,  remove_columns=["audio", "sampling_rate", "text"])

print(f"  Train: {len(train_ds)} samples")
print(f"  Eval:  {len(eval_ds)} samples")

In [ ]:
import dataclasses
from dataclasses import dataclass
from typing import Any, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch   = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        # Remove decoder_start_token if prepended
        if (labels[:, 0] == ft_model.config.decoder_start_token_id).all():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch


data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

# WER metric
wer_metric = hf_evaluate.load("wer")

def compute_metrics(pred):
    pred_ids   = pred.predictions
    label_ids  = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str  = processor.tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    wer_val = wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": round(wer_val * 100, 2)}


# ── Training arguments (lightweight — small dataset) ──
training_args = Seq2SeqTrainingArguments(
    output_dir=FINETUNE_MODEL_DIR,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    warmup_steps=5,
    max_steps=50,              # small dataset — keep low
    fp16=torch.cuda.is_available(),
    evaluation_strategy="steps",
    eval_steps=25,
    save_steps=25,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    predict_with_generate=True,
    generation_max_length=225,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=ft_model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=processor.feature_extractor,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Starting fine-tuning (50 steps)...")
train_result = trainer.train()
print("✅ Fine-tuning complete!")
print(f"Training loss: {train_result.training_loss:.4f}")

# Save fine-tuned model
trainer.save_model(FINETUNE_MODEL_DIR)
processor.save_pretrained(FINETUNE_MODEL_DIR)
print(f"Fine-tuned model saved to: {FINETUNE_MODEL_DIR}")

---
## 📈 8. Before vs After Fine-Tuning

In [ ]:
# Load fine-tuned whisper and test
print("Loading fine-tuned Whisper for evaluation...")
ft_whisper = whisper.load_model("base")  # Note: must load from HF for truly fine-tuned
# If using HF fine-tuned:
# from transformers import WhisperForConditionalGeneration, WhisperProcessor
# ft_processor = WhisperProcessor.from_pretrained(FINETUNE_MODEL_DIR)
# ft_hf_model  = WhisperForConditionalGeneration.from_pretrained(FINETUNE_MODEL_DIR)

print("Evaluating fine-tuned model on EN samples...")
ft_hyps_en = [transcribe_whisper(p, language="en") for p in EN_AUDIO_PATHS]
ft_hyps_id = [transcribe_whisper(p, language="id") for p in ID_AUDIO_PATHS]

ft_metrics_en = evaluate(EN_TRANSCRIPTS, ft_hyps_en)
ft_metrics_id = evaluate(ID_TRANSCRIPTS, ft_hyps_id)

print("\n=== Before Fine-tuning (Whisper base) ===")
print(f"  EN: WER={whisper_metrics_en['wer']}%, Accuracy={whisper_metrics_en['accuracy']}%")
print(f"  ID: WER={whisper_metrics_id['wer']}%, Accuracy={whisper_metrics_id['accuracy']}%")
print("\n=== After Fine-tuning ===")
print(f"  EN: WER={ft_metrics_en['wer']}%, Accuracy={ft_metrics_en['accuracy']}%")
print(f"  ID: WER={ft_metrics_id['wer']}%, Accuracy={ft_metrics_id['accuracy']}%")

# Comparison chart
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, lang, before, after in [
    (axes[0], "EN", whisper_metrics_en, ft_metrics_en),
    (axes[1], "ID", whisper_metrics_id, ft_metrics_id),
]:
    cats = ["WER (%)", "CER (%)", "Accuracy (%)"]
    b_vals = [before.get("wer") or 0, before.get("cer") or 0, before.get("accuracy") or 0]
    a_vals = [after.get("wer")  or 0, after.get("cer")  or 0, after.get("accuracy")  or 0]
    x = np.arange(len(cats))
    ax.bar(x - 0.2, b_vals, 0.4, label="Before", color="#9ca3af")
    ax.bar(x + 0.2, a_vals, 0.4, label="After",  color="#6366f1")
    ax.set_xticks(x); ax.set_xticklabels(cats)
    ax.set_title(f"Whisper Fine-tuning Effect ({lang})")
    ax.legend(); ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "finetune_comparison.png"), dpi=120)
plt.show()
print("Saved: outputs/finetune_comparison.png")

---
## 💾 9. Export Best Model Config

In [ ]:
config = {
    "best_model": best_model_key,
    "whisper_model_size": "base",
    "whisper_finetuned_dir": FINETUNE_MODEL_DIR,
    "vosk_model_en": VOSK_MODELS["en"]["dir"],
    "vosk_model_id": VOSK_MODELS["id"]["dir"],
    "comparison": [
        {
            "model": m,
            "wer_en": en_m.get("wer"),
            "wer_id": id_m.get("wer"),
            "acc_en": en_m.get("accuracy"),
            "acc_id": id_m.get("accuracy"),
            "avg_rtf_en": en_m.get("avg_rtf"),
        }
        for m, en_m, id_m in zip(models, en_metrics_list, id_metrics_list)
    ],
}

config_path = "model_config.json"
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

print(f"✅ Saved model_config.json")
print(json.dumps(config, indent=2))

---
## 🏁 10. Summary

| Step | Status |
|------|--------|
| Dataset download (EN) | ✅ HuggingFace multispeak |
| Dataset download (ID) | ✅ Kaggle / Common Voice fallback |
| Audio preprocessing | ✅ 16kHz mono, peak-normalized |
| Vosk inference | ✅ |
| Whisper inference | ✅ |
| Wav2Vec2 inference | ✅ |
| SpeechBrain inference | ✅ |
| Model comparison | ✅ outputs/model_comparison.png |
| Fine-tuning (best model) | ✅ models/whisper-finetuned/ |
| Config export | ✅ model_config.json |

**Run the Streamlit app:**
```bash
streamlit run app.py
```

The app will automatically read `model_config.json` to use the best model for final transcription.